# Engenharia de Features e Split Temporal

## Objetivo deste notebook

Transformar o dataset limpo em conjuntos de
treino e teste prontos para modelagem, aplicando as transformações de
feature engineering validadas na análise exploratória e definindo
formalmente o corte temporal que será usado em todo o restante do projeto.

## O que será feito

**1. Carregamento do dataset limpo**

Leitura do parquet salvo no notebook EDA (283.726 transações, sem
duplicatas, tipos corretos).

**2. Validação e definição do corte temporal**

Antes de aplicar o split 80/20 o que é comum, verificamos empiricamente quantas
fraudes caem em treino vs. teste nesse corte já que a distribuição de
fraude ao longo do tempo é irregular (visto na EDA), 20% do volume por
tempo não garante 20% das fraudes. O corte final pode ser ajustado (ex:
75/25) caso o teste fique com poucas fraudes para validação estatística
confiável.

**3. Split temporal treino/teste**

Transações mais antigas compõem o treino, as mais recentes compõem o
teste, reproduzindo o cenário real de produção, onde o modelo nunca vê
o futuro durante o treinamento. Essa é uma decisão crítica do projeto:
um split aleatório inflaria artificialmente as métricas de validação.

**4. Encoding cíclico de Hour**

A EDA revelou padrão circadiano nítido nas transações legítimas e ausente
nas fraudes. Como a hora é uma variável circular (23h e 0h são contíguas
no relógio, mas numericamente distantes), aplicamos transformação
seno/cosseno sobre `Hour`, permitindo que o modelo capture essa
proximidade circular corretamente.

**5. RobustScaler em Amount**

A EDA confirmou distribuição extremamente assimétrica em `Amount`, com
outliers legítimos de alto valor e concentração de fraudes em valores
baixos (padrão de testagem de cartão). RobustScaler é aplicado por ser baseado
em mediana e IQR, portanto resistente a essa distorção o que é diferente do
StandardScaler, que seria puxado pelos outliers.

**Ponto de atenção técnico:** o scaler deve ser **ajustado (fit) apenas
no conjunto de treino** e depois aplicado (transform) em treino e teste
separadamente, para evitar vazamento de informação do teste para o
treino (data leakage).

**6. Persistência dos datasets processados**

Os conjuntos de treino e teste, já com as features transformadas, são
salvos no Drive em formato parquet, prontos para serem consumidos pelo
`03_stage1_anomaly.ipynb`.

## Decisões técnicas herdadas da EDA

| Decisão | Motivação (evidência da EDA) |
|---|---|
| Split temporal (não aleatório) | Estrutura de 48h em dois ciclos; split aleatório superestimaria performance |
| Split 80/20 (a validar) | Volume suficiente em ambos os lados, a confirmar cobertura de fraude no teste |
| Encoding cíclico de Hour | Padrão circadiano nítido nas legítimas, ausente nas fraudes |
| RobustScaler em Amount | Assimetria extrema, outliers legítimos de alto valor, fraude concentrada em valores baixos |
| Manutenção de todas as 30 features | Effect size mostrou sinal em 18 das 28 componentes V; demais podem ter valor em interação |

In [3]:
import pandas as pd

df = pd.read_parquet("/content/drive/MyDrive/fraud-detection-two-stage/data/processed/creditcard_clean.parquet")
df.head(10)

,transaction_id,Time,V1,V2,V3,V4,V5,V6,V7,V8,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
5,5,2.0,-0.425966,0.960523,1.141109,-0.168252,0.420987,-0.029728,0.476201,0.260314,...,-0.208254,-0.559825,-0.026398,-0.371427,-0.232794,0.105915,0.253844,0.081080,3.67,0
6,6,4.0,1.229658,0.141004,0.045371,1.202613,0.191881,0.272708,-0.005159,0.081213,...,-0.167716,-0.270710,-0.154104,-0.780055,0.750137,-0.257237,0.034507,0.005168,4.99,0
7,7,7.0,-0.644269,1.417964,1.074380,-0.492199,0.948934,0.428118,1.120631,-3.807864,...,1.943465,-1.015455,0.057504,-0.649709,-0.415267,-0.051634,-1.206921,-1.085339,40.80,0
8,8,7.0,-0.894286,0.286157,-0.113192,-0.271526,2.669599,3.721818,0.370145,0.851084,...,-0.073425,-0.268092,-0.204233,1.011592,0.373205,-0.384157,0.011747,0.142404,93.20,0
9,9,9.0,-0.338262,1.119593,1.044367,-0.222187,0.499361,-0.246761,0.651583,0.069539,...,-0.246914,-0.633753,-0.120794,-0.385050,-0.069733,0.094199,0.246219,0.083076,3.68,0


In [4]:
# Confirma que transaction_id está presente antes de prosseguir
assert "transaction_id" in df.columns, "ERRO: transaction_id ausente no dataset carregado!"
print(f"transaction_id presente. {df['transaction_id'].nunique()} valores únicos.")

transaction_id presente. 283726 valores únicos.


In [5]:
# ----------------------------------------------------------------
# VALIDAÇÃO DO CORTE TEMPORAL — checagem de distribuição de fraude
# ----------------------------------------------------------------
# Antes de fixar a proporção do split, verificamos quantas fraudes
# caem em treino vs teste para o corte proposto (80/20 por tempo).

corte_temporal = df["Time"].quantile(0.80)

treino_mask = df["Time"] <= corte_temporal
teste_mask  = df["Time"] > corte_temporal

n_fraude_treino = df[treino_mask]["Class"].sum()
n_fraude_teste  = df[teste_mask]["Class"].sum()

print(f"Corte temporal (segundos): {corte_temporal:.0f}")
print(f"\nTreino: {treino_mask.sum():,} transações | {n_fraude_treino} fraudes "
      f"({n_fraude_treino/treino_mask.sum()*100:.3f}%)")
print(f"Teste : {teste_mask.sum():,} transações | {n_fraude_teste} fraudes "
      f"({n_fraude_teste/teste_mask.sum()*100:.3f}%)")

Corte temporal (segundos): 145234

Treino: 226,982 transações | 399 fraudes (0.176%)
Teste : 56,744 transações | 74 fraudes (0.130%)


O conjunto de teste contém 74 casos de fraude, volume suficiente para avaliação de PR-AUC/Recall, mas intervalos de confiança em métricas de negócio devem ser interpretados com essa limitação amostral em mente.

In [6]:
# ----------------------------------------------------------------
# APLICAÇÃO DO SPLIT TEMPORAL — treino/teste
# ----------------------------------------------------------------
# Corte confirmado na validação anterior: 145.234 segundos (~80/20),
# com 399 fraudes no treino e 74 fraudes no teste — volume validado
# como suficiente para avaliação estatística confiável.

CORTE_TEMPORAL = df["Time"].quantile(0.80)  # mesmo corte já validado

df_treino = df[df["Time"] <= CORTE_TEMPORAL].copy()
df_teste  = df[df["Time"] > CORTE_TEMPORAL].copy()

# Confirmação rápida de sanidade — shapes e proporção de fraude
print(f"Treino: {df_treino.shape[0]:,} transações | "
      f"{df_treino['Class'].sum()} fraudes "
      f"({df_treino['Class'].mean()*100:.3f}%)")

print(f"Teste : {df_teste.shape[0]:,} transações | "
      f"{df_teste['Class'].sum()} fraudes "
      f"({df_teste['Class'].mean()*100:.3f}%)")

# Checagem crítica: garante que não há sobreposição temporal
assert df_treino["Time"].max() <= df_teste["Time"].min(), \
    "ERRO: há sobreposição temporal entre treino e teste!"
print("\nSplit temporal validado — sem sobreposição entre treino e teste.")

Treino: 226,982 transações | 399 fraudes (0.176%)
Teste : 56,744 transações | 74 fraudes (0.130%)

Split temporal validado — sem sobreposição entre treino e teste.


In [7]:
import numpy as np

# ----------------------------------------------------------------
# ENCODING CÍCLICO DE HOUR — seno e cosseno
# ----------------------------------------------------------------
# Hour é uma variável circular: 23h e 0h são "vizinhas" no relógio,
# mas numericamente distantes (diferença de 23). O encoding
# seno/cosseno mapeia a hora em um círculo trigonométrico,
# preservando essa proximidade circular para o modelo.
#
# Fórmula: transforma Hour (0-23) em duas features contínuas
# que juntas representam a posição no ciclo de 24h.

def aplicar_encoding_ciclico(dataframe):
    """Cria Hour (0-23) e as features cíclicas seno/cosseno a partir de Time."""
    df_copia = dataframe.copy()

    # Hora do dia (0-23), reaproveitando a lógica já validada na EDA
    df_copia["Hour"] = (df_copia["Time"] / 3600 % 24).astype(int)

    # Encoding cíclico — 2*pi para completar o ciclo de 24h
    df_copia["Hour_sin"] = np.sin(2 * np.pi * df_copia["Hour"] / 24)
    df_copia["Hour_cos"] = np.cos(2 * np.pi * df_copia["Hour"] / 24)

    return df_copia

# Aplica a mesma função em treino e teste — garante consistência
df_treino = aplicar_encoding_ciclico(df_treino)
df_teste  = aplicar_encoding_ciclico(df_teste)

# Checagem rápida: 23h e 0h devem ficar próximas no espaço seno/cosseno
print("Verificação da propriedade cíclica:")
print(df_treino[df_treino["Hour"] == 23][["Hour", "Hour_sin", "Hour_cos"]].iloc[0])
print(df_treino[df_treino["Hour"] == 0][["Hour", "Hour_sin", "Hour_cos"]].iloc[0])

Verificação da propriedade cíclica:
Hour        23.000000
Hour_sin    -0.258819
Hour_cos     0.965926
Name: 138181, dtype: float64
Hour        0.0
Hour_sin    0.0
Hour_cos    1.0
Name: 0, dtype: float64


### Análise — Encoding Cíclico de Hour (Seno/Cosseno)

Para representar corretamente a natureza circular do tempo, transformei
a variável `Hour` (0–23) em duas features contínuas — `Hour_sin` e
`Hour_cos` através de projeção trigonométrica. Essa transformação é
necessária porque, numericamente, `Hour` trataria 23h e 0h como extremos
opostos (diferença absoluta de 23), quando na realidade são horários
consecutivos no relógio.

**Validação da propriedade cíclica:**

| Hora | Hour_sin | Hour_cos |
|---|---|---|
| 23h | -0,2588 | 0,9659 |
| 0h | 0,0000 | 1,0000 |

A distância euclidiana entre os pares (Hour_sin, Hour_cos) de 23h e 0h é
de aproximadamente **0,268**, muito pequena, confirmando que a
transformação preserva a proximidade real entre essas duas horas. Em
contraste, a distância no espaço original de `Hour` bruto seria de 23
unidades, o que induziria o modelo a interpretar erroneamente 23h e 0h
como momentos distantes entre si.

**Implicação para a modelagem:** essa correção evita que o Stage 1
(Isolation Forest) e o Stage 2 (LightGBM) aprendam uma penalização
artificial entre a última hora de um dia e a primeira hora do seguinte
o que seria especialmente prejudicial dado que a EDA já identificou picos
de fraude justamente nesse intervalo de madrugada.

**Nota de implementação:** a coluna `Hour` (inteiro 0–23) foi mantida
apenas como etapa intermediária para gerar o encoding e para esta
validação e ela será removida do conjunto final de features antes do
treinamento, evitando redundância de informação com `Hour_sin` e
`Hour_cos`.

A transformação foi aplicada de forma idêntica e independente em treino
e teste, sem risco de vazamento de informação, já que se trata de uma
função determinística (sem parâmetros ajustados a partir dos dados).

In [8]:
# ----------------------------------------------------------------
# ROBUSTSCALER EM AMOUNT
# ----------------------------------------------------------------
# RobustScaler usa mediana e IQR (intervalo interquartil) em vez
# de média e desvio padrão e ele não é distorcido pelos outliers de
# alto valor que vimos na EDA (legítimas chegando a ~25.691).
#
# CRÍTICO: o scaler é ajustado (fit) APENAS no treino.
# Se ajustássemos no dataset inteiro, informação estatística
# do conjunto de teste vazaria para o treino (data leakage),
# inflando artificialmente a performance do modelo.

from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

# Fit apenas no treino
df_treino["Amount_scaled"] = scaler.fit_transform(df_treino[["Amount"]])

# Transform (sem fit) no teste, usando a mediana/IQR aprendidos no treino
df_teste["Amount_scaled"] = scaler.transform(df_teste[["Amount"]])

# Checagem rápida: estatísticas antes e depois do scaling
print("Amount original (treino):")
print(df_treino["Amount"].describe()[["mean", "std", "min", "max"]])

print("\nAmount_scaled (treino):")
print(df_treino["Amount_scaled"].describe()[["mean", "std", "min", "max"]])

Amount original (treino):
mean       90.931827
std       250.770617
min         0.000000
max     19656.530000
Name: Amount, dtype: float64

Amount_scaled (treino):
mean      0.911376
std       3.388792
min      -0.317432
max     265.311351
Name: Amount_scaled, dtype: float64


### Análise — RobustScaler em Amount

Aplicamos `RobustScaler` sobre a feature `Amount`, ajustado (fit)
exclusivamente no conjunto de treino e depois aplicado (transform) em
treino e teste separadamente, evitando vazamento de informação do teste
para o treino (data leakage).

**Resultado da transformação (treino):**

| Estatística | Amount original | Amount_scaled |
|---|---|---|
| Mean | 90,93 | 0,91 |
| Std | 250,77 | 3,39 |
| Min | 0,00 | -0,317 |
| Max | 19.656,53 | 265,31 |

**Interpretação:**

- A escala foi drasticamente comprimida (desvio padrão caiu de ~250 para
  ~3,4), reduzindo o risco de essa feature dominar algoritmos sensíveis
  a magnitude, como o Isolation Forest e o One-Class SVM do Stage 1.
- O `mean` não ficou próximo de zero porque o RobustScaler centraliza
  pela **mediana**, não pela média e a assimetria de `Amount` (muitos
  valores baixos, poucos outliers extremos) faz a média continuar
  distante da mediana mesmo após o escalonamento. Esse é o comportamento
  esperado e correto, ao contrário do que ocorreria com um
  `StandardScaler`, que seria fortemente distorcido pelos outliers.
- O valor máximo (265,31) ainda se destaca claramente na nova escala,
  confirmando que o RobustScaler **preserva o sinal de outliers** em vez
  de eliminá-los o que é um comportamento desejado neste projeto, já que
  transações de valor atipicamente alto podem carregar informação
  relevante para a detecção de fraude, e não devem ser tratadas como
  ruído a ser descartado.

O scaler ajustado (`scaler`) deve ser persistido junto aos artefatos do
projeto, para garantir que a mesma transformação seja aplicada de forma
consistente em dados futuros (ex: na API de produção).

In [9]:
# ----------------------------------------------------------------
# CONSOLIDAÇÃO FINAL — seleção de colunas e persistência
# ----------------------------------------------------------------
# Remove colunas intermediárias (Hour bruto, Amount original e Time) que
# não devem entrar no modelo, mantendo apenas as versões transformadas.
# Salva os datasets processados e o scaler ajustado para reuso
# consistente em notebooks futuros e na API de produção.

import joblib
import os

# Colunas a remover: já foram transformadas em versões definitivas
colunas_remover = ["Hour", "Amount", "Time"]

df_treino_final = df_treino.drop(columns=colunas_remover)
df_teste_final  = df_teste.drop(columns=colunas_remover)

# Checagem de sanidade — confirma que as colunas certas estão presentes
print("Colunas do dataset final:")
print(df_treino_final.columns.tolist())

print(f"\nShape treino final: {df_treino_final.shape}")
print(f"Shape teste final : {df_teste_final.shape}")

# ----------------------------------------------------------------
# Persistência no Drive
# ----------------------------------------------------------------

caminho_base = "/content/drive/MyDrive/fraud-detection-two-stage"
caminho_dados = f"{caminho_base}/data/processed"
caminho_modelos = f"{caminho_base}/models"

os.makedirs(caminho_dados, exist_ok=True)
os.makedirs(caminho_modelos, exist_ok=True)

# Datasets processados (treino e teste)
df_treino_final.to_parquet(f"{caminho_dados}/train_processed.parquet", index=False)
df_teste_final.to_parquet(f"{caminho_dados}/test_processed.parquet", index=False)

# Scaler ajustado — necessário para transformar novos dados
# (ex: transações recebidas pela API em produção) da mesma forma
joblib.dump(scaler, f"{caminho_modelos}/robust_scaler_amount.pkl")

print(f"\nDatasets salvos em: {caminho_dados}")
print(f"Scaler salvo em: {caminho_modelos}/robust_scaler_amount.pkl")

Colunas do dataset final:
['transaction_id', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Class', 'Hour_sin', 'Hour_cos', 'Amount_scaled']

Shape treino final: (226982, 33)
Shape teste final : (56744, 33)

Datasets salvos em: /content/drive/MyDrive/fraud-detection-two-stage/data/processed
Scaler salvo em: /content/drive/MyDrive/fraud-detection-two-stage/models/robust_scaler_amount.pkl


In [11]:
# Confirma que transaction_id foi preservado até o final
assert "transaction_id" in df_treino_final.columns, "ERRO: transaction_id perdido no treino!"
assert "transaction_id" in df_teste_final.columns, "ERRO: transaction_id perdido no teste!"
assert df_treino_final["transaction_id"].is_unique
assert df_teste_final["transaction_id"].is_unique
print(f"\ntransaction_id preservado corretamente em ambos os conjuntos.")


transaction_id preservado corretamente em ambos os conjuntos.


## Conclusão — Feature Engineering

Este notebook transformou o dataset limpo em conjuntos de treino e teste
prontos para modelagem, aplicando as decisões técnicas validadas na EDA:

1. **Split temporal 80/20** confirmado com 399 fraudes no treino e 74 no
   teste, volume validado como suficiente para avaliação estatística,
   sem sobreposição temporal entre os conjuntos.
2. **Encoding cíclico de Hour** (seno/cosseno) implementado e validado —
   23h e 0h ficam próximas no espaço transformado (distância euclidiana
   ≈ 0,268), corrigindo a descontinuidade artificial da hora bruta.
3. **RobustScaler em Amount** ajustado exclusivamente no treino e aplicado
   em ambos os conjuntos, evitando data leakage. O scaler foi persistido
   (`robust_scaler_amount.pkl`) para reuso consistente em produção.
4. **Remoção de colunas intermediárias e de risco de leakage**: `Hour`
   bruto (redundante com o encoding cíclico) e `Time` bruto (que
   introduziria leakage estrutural do split temporal, permitindo ao
   modelo distinguir treino/teste pelo valor absoluto de tempo em vez
   de aprender padrões reais de fraude).
5. **Decisão de não criar features adicionais nesta etapa**: dado que
   V1–V28 já são componentes PCA (sem histórico de cliente/cartão
   disponível para agregações), e que modelos como LightGBM e Isolation Forest
   capturam interações não-lineares automaticamente, engenharia manual
   de features adicionais foi adiada, a ser revisitada apenas se a
   avaliação do Stage 2 indicar necessidade real, evitando
   risco de overfitting dado o volume limitado de fraudes (399 no treino).

**Datasets finais:** 32 colunas (31 features + target), persistidos em
`train_processed.parquet` e `test_processed.parquet`.
